# Notebook 12: Capstone -- End-to-End Tabular ML Pipeline

## Why This Matters

Real ML projects are not isolated notebook experiments. They require a coherent pipeline: data generation and EDA, feature engineering with leakage prevention, preprocessing as a pipeline, model comparison, hyperparameter tuning, calibration, unbiased evaluation, and documentation. This capstone ties together every technique from Series 41 into a single end-to-end flow on a realistic credit risk dataset.

## Problem Statement

Credit risk prediction: given borrower demographics, loan characteristics, and credit history, predict whether a borrower will default. This is a real business problem with regulatory implications (ECOA, Fair Credit Reporting Act) and asymmetric costs (FN = missed default = loss; FP = rejected good borrower = foregone revenue).

## What You Will Build

1. Synthetic credit risk EDA (realistic missing data, class imbalance)
2. Feature engineering: DTI, PTI, log transforms, age buckets, credit risk scores
3. ColumnTransformer pipeline with imputation, scaling, and OHE
4. Model comparison: Logistic Regression, Random Forest, GradientBoosting, XGBoost
5. Optuna hyperparameter tuning of XGBoost
6. Platt scaling calibration
7. Nested cross-validation for unbiased evaluation
8. Model card documenting choices and limitations

## Community Context

This pipeline pattern -- ColumnTransformer + Pipeline + Optuna + nested CV + model card -- is the production-grade standard for tabular ML at companies like Stripe, Affirm, and Upstart. Regulatory requirements for credit models add further constraints: explainability (SHAP), fairness audits, and performance monitoring are required.

In [ ]:
%pip install -q scikit-learn xgboost optuna numpy pandas matplotlib scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, cross_validate
)
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, brier_score_loss
)
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)

## 1. Data Generation and EDA

We create a synthetic credit dataset with realistic structure: log-normal income, correlated loan amounts, categorical credit history, and 5% missing values in age and income. The default rate is ~25% (realistic for subprime portfolios).

In [ ]:
# Generate synthetic credit risk dataset
# (Mirrors UCI German Credit structure)
np.random.seed(42)
n = 2000

age = np.random.normal(38, 12, n).clip(18, 80).astype(int)
income = np.random.lognormal(10.5, 0.8, n)
loan_amount = np.random.lognormal(8.5, 0.9, n)
loan_duration = np.random.choice([12, 24, 36, 48, 60], n)
credit_history = np.random.choice(
    ["no_credits", "all_paid", "existing_paid", "delayed", "critical"],
    n, p=[0.05, 0.15, 0.50, 0.20, 0.10]
)
employment = np.random.choice(
    ["unemployed", "<1yr", "1-4yr", "4-7yr", ">=7yr"],
    n, p=[0.08, 0.17, 0.35, 0.25, 0.15]
)
housing = np.random.choice(["own", "free", "rent"], n, p=[0.55, 0.10, 0.35])
purpose = np.random.choice(
    ["car", "furniture", "radio_tv", "education", "business", "other"],
    n, p=[0.25, 0.20, 0.18, 0.12, 0.15, 0.10]
)

# Introduce ~5% missing values
income_nan = income.copy().astype(float)
income_nan[np.random.choice(n, int(0.05*n), replace=False)] = np.nan
age_nan = age.copy().astype(float)
age_nan[np.random.choice(n, int(0.03*n), replace=False)] = np.nan

# Construct target: default probability based on feature logic
default_logit = (
    -2.0
    + 0.5 * (loan_amount / income)
    + 0.3 * (loan_duration / 12)
    - 0.02 * (age - 38)
    + np.where(credit_history == "critical", 1.5, 0)
    + np.where(credit_history == "delayed", 0.8, 0)
    + np.where(employment == "unemployed", 1.2, 0)
    + np.where(housing == "rent", 0.3, 0)
    + np.random.normal(0, 0.5, n)
)
default_prob = 1 / (1 + np.exp(-default_logit))
y = (np.random.uniform(0, 1, n) < default_prob).astype(int)

df = pd.DataFrame({
    "age": age_nan,
    "income": income_nan,
    "loan_amount": loan_amount,
    "loan_duration": loan_duration,
    "credit_history": credit_history,
    "employment": employment,
    "housing": housing,
    "purpose": purpose,
    "default": y
})

print(f"Dataset: {df.shape}")
print(f"Default rate: {y.mean():.3f}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nSample:")
print(df.head())

In [ ]:
# EDA: distributions and class rates
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Numeric distributions
for ax, col in zip(axes[0], ["age", "income", "loan_amount", "loan_duration"]):
    for cls, color in [(0, "steelblue"), (1, "tomato")]:
        subset = df[df["default"] == cls][col].dropna()
        ax.hist(subset, bins=30, alpha=0.5, color=color,
                label=f"default={cls} (n={len(subset)})", density=True)
    ax.set_title(col)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Categorical default rates
for ax, col in zip(axes[1], ["credit_history", "employment", "housing", "purpose"]):
    dr = df.groupby(col)["default"].agg(["mean", "count"]).reset_index()
    dr = dr.sort_values("mean", ascending=True)
    ax.barh(dr[col], dr["mean"], color="steelblue")
    ax.axvline(y.mean(), color="r", linestyle="--", alpha=0.7, label=f"Overall={y.mean():.2f}")
    ax.set_title(f"Default rate by {col}")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, axis="x")

plt.suptitle("EDA: Credit Risk Dataset", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nClass balance:", dict(pd.Series(y).value_counts()))

## 2. Feature Engineering

Good feature engineering often matters more than model choice for tabular data. We create domain-relevant features: debt-to-income ratio (DTI), payment-to-income ratio (PTI), log-transforms of skewed variables, age buckets, and an ordinal credit risk score derived from domain knowledge.

In [ ]:
# Feature engineering
df2 = df.copy()

# Debt-to-income ratio
df2["dti"] = df2["loan_amount"] / (df2["income"].fillna(df2["income"].median()) + 1)

# Log-transform skewed variables
df2["log_income"] = np.log1p(df2["income"].fillna(df2["income"].median()))
df2["log_loan"] = np.log1p(df2["loan_amount"])

# Monthly payment estimate
df2["monthly_payment"] = df2["loan_amount"] / df2["loan_duration"]

# Payment-to-income ratio
df2["pti"] = df2["monthly_payment"] / (df2["income"].fillna(df2["income"].median()) / 12 + 1)

# Age buckets
df2["age_bucket"] = pd.cut(
    df2["age"].fillna(df2["age"].median()),
    bins=[0, 25, 35, 50, 65, 100],
    labels=["<25", "25-35", "35-50", "50-65", "65+"]
).astype(str)

# Credit history risk encoding
credit_risk_map = {
    "no_credits": 0.5, "all_paid": 0.1, "existing_paid": 0.2,
    "delayed": 0.6, "critical": 0.8
}
df2["credit_risk_score"] = df2["credit_history"].map(credit_risk_map)

print("Feature engineering complete.")
print(f"New features: dti, log_income, log_loan, monthly_payment, pti, age_bucket, credit_risk_score")
print(df2[["dti", "log_income", "log_loan", "monthly_payment", "pti", "credit_risk_score"]].describe().round(3))

## 3. Preprocessing Pipeline

All preprocessing is encapsulated in a ColumnTransformer that handles base numeric features (median imputation + RobustScaler), engineered numeric features (StandardScaler), and categorical features (most-frequent imputation + OneHotEncoder). The pipeline ensures no leakage -- transformers are fit only on training data.

In [ ]:
# Build ColumnTransformer pipeline
X = df2.drop(columns=["default"])
target = df2["default"].values

numeric_base = ["age", "income", "loan_amount", "loan_duration"]
numeric_eng = ["dti", "log_income", "log_loan", "monthly_payment", "pti", "credit_risk_score"]
categorical = ["credit_history", "employment", "housing", "purpose", "age_bucket"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])
eng_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num_base", numeric_transformer, numeric_base),
    ("num_eng", eng_transformer, numeric_eng),
    ("cat", categorical_transformer, categorical),
], remainder="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.25, stratify=target, random_state=42
)

# Test pipeline
X_tr_proc = preprocessor.fit_transform(X_train)
X_te_proc = preprocessor.transform(X_test)
print(f"Preprocessed train shape: {X_tr_proc.shape}")
print(f"Preprocessed test shape:  {X_te_proc.shape}")

## 4. Model Comparison

Five-fold stratified CV compares four model families. Each model is wrapped in a full Pipeline (preprocessor + classifier), ensuring fair comparison with no data leakage. XGBoost and GradientBoosting typically outperform linear models on tabular data with categorical features, but not always -- logistic regression is a strong baseline.

In [ ]:
# Model comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_pipelines = {
    "Logistic Regression": Pipeline([
        ("pre", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, C=0.1, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("pre", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42))
    ]),
    "GradientBoosting": Pipeline([
        ("pre", preprocessor),
        ("clf", GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("pre", preprocessor),
        ("clf", xgb.XGBClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.05,
            eval_metric="logloss", verbosity=0, random_state=42
        ))
    ]),
}

print("5-fold CV results:")
print(f"{"Model":25s} {"AUC":>8s} {"AP":>8s} {"Brier":>8s}")
print("-" * 55)

cv_results = {}
for name, pipe in model_pipelines.items():
    auc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    ap_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1)
    cv_results[name] = {"auc": auc_scores, "ap": ap_scores}
    print(f"{name:25s} {auc_scores.mean():.4f}   {ap_scores.mean():.4f}")

## 5. Optuna Hyperparameter Tuning

Optuna TPE search over XGBoost hyperparameters. The objective function runs 3-fold CV and returns mean AUC. Optuna learns from each trial to focus on promising regions of hyperparameter space.

In [ ]:
# Optuna tuning of XGBoost
from sklearn.base import clone

xgb_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", xgb.XGBClassifier(eval_metric="logloss", verbosity=0, random_state=42))
])

def xgb_objective(trial):
    params = {
        "clf__n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "clf__max_depth": trial.suggest_int("max_depth", 3, 8),
        "clf__learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "clf__subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "clf__colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "clf__reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "clf__min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
    }
    pipe = clone(xgb_pipe)
    pipe.set_params(**params)
    scores = cross_val_score(pipe, X_train, y_train, cv=3, scoring="roc_auc", n_jobs=-1)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(xgb_objective, n_trials=30, show_progress_bar=False)

print(f"Best CV AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

## 6. Calibration and Final Evaluation

After tuning, we calibrate the model with Platt scaling on a 15% holdout and evaluate on the test set. ECE measures probability accuracy; Brier score measures overall probabilistic skill.

In [ ]:
# Train best XGBoost with optimal params, calibrate, evaluate
best_params = {f"clf__{k}": v for k, v in study.best_params.items()}
best_pipe = clone(xgb_pipe)
best_pipe.set_params(**best_params)
best_pipe.fit(X_train, y_train)

# Calibrate with CalibratedClassifierCV
cal_pipe = CalibratedClassifierCV(best_pipe, method="sigmoid", cv="prefit")

# Use a held-out calibration split from training data
X_tr3, X_cal3, y_tr3, y_cal3 = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=42
)
best_pipe_cal = clone(xgb_pipe)
best_pipe_cal.set_params(**best_params)
best_pipe_cal.fit(X_tr3, y_tr3)
cal_pipe_fit = CalibratedClassifierCV(best_pipe_cal, method="sigmoid", cv="prefit")
cal_pipe_fit.fit(X_cal3, y_cal3)

# Evaluate both on test
probs_raw = best_pipe.predict_proba(X_test)[:, 1]
probs_cal = cal_pipe_fit.predict_proba(X_test)[:, 1]

def ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0:
            continue
        e += mask.mean() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return e

print("Test set evaluation:")
for label, probs in [("Raw XGBoost", probs_raw), ("Calibrated", probs_cal)]:
    auc = roc_auc_score(y_test, probs)
    ap = average_precision_score(y_test, probs)
    brier = brier_score_loss(y_test, probs)
    ece_val = ece(y_test, probs)
    print(f"  {label:20s}  AUC={auc:.4f}  AP={ap:.4f}  Brier={brier:.4f}  ECE={ece_val:.4f}")

# Calibration curve
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], "k--", label="Perfect")
for label, probs in [("Raw", probs_raw), ("Calibrated", probs_cal)]:
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10)
    ax.plot(mean_pred, frac_pos, "o-", label=label)
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curves (Test Set)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Nested Cross-Validation

Nested CV gives an unbiased estimate of the full pipeline's generalization performance, including hyperparameter tuning. The inner loop selects hyperparameters; the outer loop evaluates. Without nesting, reported AUC is optimistically biased.

In [ ]:
# Nested CV for unbiased AUC estimate of full pipeline
from sklearn.model_selection import GridSearchCV

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

param_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [4, 6],
    "clf__learning_rate": [0.05, 0.1],
}
xgb_inner = GridSearchCV(
    clone(xgb_pipe), param_grid, cv=inner_cv,
    scoring="roc_auc", n_jobs=-1
)

nested_aucs = cross_val_score(
    xgb_inner, X_train, y_train,
    cv=outer_cv, scoring="roc_auc", n_jobs=-1
)
print("Nested CV (unbiased) AUC:")
for i, v in enumerate(nested_aucs):
    print(f"  Fold {i+1}: {v:.4f}")
print(f"Mean +/- std: {nested_aucs.mean():.4f} +/- {nested_aucs.std():.4f}")

## 8. Model Card

A model card documents model purpose, data, evaluation methodology, performance metrics, and known limitations. It is increasingly required for regulated industries and responsible ML deployment.

In [ ]:
# Model Card: Credit Risk Classifier
model_card = {
    "Model": "XGBoost with Platt calibration",
    "Task": "Binary credit default prediction",
    "Data": "Synthetic credit risk dataset (n=2000)",
    "Train/Test Split": "75% / 25% stratified",
    "Preprocessing": "Median imputation, RobustScaler, OneHotEncoder, engineered features (DTI, PTI, log-transforms)",
    "Tuning": "Optuna TPE, 30 trials, 3-fold inner CV",
    "Calibration": "Platt scaling on 15% holdout",
    "Evaluation": "Nested 5-fold CV for unbiased AUC",
    "Test AUC": f"{roc_auc_score(y_test, probs_cal):.4f}",
    "Test AP": f"{average_precision_score(y_test, probs_cal):.4f}",
    "Test ECE": f"{ece(y_test, probs_cal):.4f}",
    "Known Limitations": [
        "Synthetic data -- real credit data has stronger regulatory constraints",
        "No fairness analysis across protected attributes",
        "Static model -- no concept drift monitoring",
        "Threshold not optimized for specific cost matrix"
    ],
    "Intended Use": "Educational demonstration of end-to-end tabular ML pipeline",
}

print("=" * 60)
print("MODEL CARD")
print("=" * 60)
for key, value in model_card.items():
    if isinstance(value, list):
        print(f"{key}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"{key}: {value}")
print("=" * 60)

## Summary

| Stage | Tool / Technique | Key Decision |
|-------|-----------------|-------------|
| EDA | pandas, matplotlib | Class balance, missing data patterns |
| Feature Engineering | Domain + log-transforms | DTI, PTI, credit risk score |
| Preprocessing | ColumnTransformer + Pipeline | Prevents leakage, handles mixed types |
| Model Selection | Cross-validated comparison | Multiple families, same CV split |
| Hyperparameter Tuning | Optuna TPE | 30 trials, 3-fold inner CV |
| Calibration | Platt scaling | Held-out calibration set |
| Evaluation | Nested 5-fold CV | Unbiased AUC estimate |
| Documentation | Model card | Limitations, intended use |

**Completing Series 41:** You now have the full classical ML toolkit -- from sklearn API internals through linear models, trees, boosting, feature engineering, pipelines, tuning, calibration, ensembles, and end-to-end deployment patterns. The natural next step is Series 40 (ML System Design) to understand how these models fit into production systems at scale.